In [ ]:
# =========================================================
# TASK 4 - STATISTICAL MODELING & RISK-BASED PRICING
# =========================================================

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBRegressor
from xgboost import XGBClassifier

import shap

import warnings
warnings.filterwarnings("ignore")

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv("../data/cleaned_data.csv")

print("Dataset Shape:", df.shape)

# =========================================================
# INITIAL INSPECTION
# =========================================================

print(df.head())

print(df.info())

print(df.isnull().sum())

# =========================================================
# FEATURE ENGINEERING
# =========================================================

# Convert dates
if "TransactionMonth" in df.columns:
    df["TransactionMonth"] = pd.to_datetime(df["TransactionMonth"])

# Example vehicle age feature
if "RegistrationYear" in df.columns:
    df["VehicleAge"] = 2025 - df["RegistrationYear"]

# Claim indicator
df["HasClaim"] = np.where(df["TotalClaims"] > 0, 1, 0)

# Margin
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

# =========================================================
# REMOVE EXTREME OUTLIERS (OPTIONAL)
# =========================================================

q1 = df["TotalClaims"].quantile(0.25)
q3 = df["TotalClaims"].quantile(0.75)

iqr = q3 - q1

upper = q3 + 1.5 * iqr

df = df[df["TotalClaims"] <= upper]

# =========================================================
# CLAIM SEVERITY DATASET
# ONLY POLICIES WITH CLAIMS
# =========================================================

severity_df = df[df["TotalClaims"] > 0].copy()

# =========================================================
# SELECT FEATURES
# =========================================================

target = "TotalClaims"

drop_cols = [
    "TotalClaims",
    "HasClaim"
]

features = [
    col for col in severity_df.columns
    if col not in drop_cols
]

X = severity_df[features]
y = severity_df[target]

# =========================================================
# IDENTIFY COLUMN TYPES
# =========================================================

categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

# =========================================================
# PREPROCESSING PIPELINE
# =========================================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# =========================================================
# MODELS
# =========================================================

models = {
    "Linear Regression": LinearRegression(),
    
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),
    
    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    )
}

# =========================================================
# TRAIN & EVALUATE REGRESSION MODELS
# =========================================================

results = []

trained_models = {}

for name, model in models.items():
    
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    
    pipeline.fit(X_train, y_train)
    
    preds = pipeline.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    r2 = r2_score(y_test, preds)
    
    results.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2
    })
    
    trained_models[name] = pipeline
    
    print("=" * 50)
    print(name)
    print("RMSE:", rmse)
    print("R2:", r2)

# =========================================================
# MODEL COMPARISON TABLE
# =========================================================

results_df = pd.DataFrame(results)

print("\nMODEL COMPARISON")
print(results_df)

# =========================================================
# VISUALIZE MODEL PERFORMANCE
# =========================================================

plt.figure(figsize=(10,5))

sns.barplot(
    data=results_df,
    x="Model",
    y="RMSE"
)

plt.title("RMSE Comparison")
plt.show()

plt.figure(figsize=(10,5))

sns.barplot(
    data=results_df,
    x="Model",
    y="R2"
)

plt.title("R² Comparison")
plt.show()

# =========================================================
# CLAIM PROBABILITY MODEL
# =========================================================

classification_df = df.copy()

X_class = classification_df.drop(
    columns=["HasClaim", "TotalClaims"]
)

y_class = classification_df["HasClaim"]

categorical_cols_class = X_class.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols_class = X_class.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

preprocessor_class = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numerical_cols_class
        ),
        
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols_class
        )
    ]
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    random_state=42
)

classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor_class),
        (
            "model",
            XGBClassifier(
                n_estimators=100,
                learning_rate=0.1,
                max_depth=5,
                random_state=42
            )
        )
    ]
)

classifier.fit(X_train_c, y_train_c)

class_preds = classifier.predict(X_test_c)

accuracy = accuracy_score(y_test_c, class_preds)
precision = precision_score(y_test_c, class_preds)
recall = recall_score(y_test_c, class_preds)
f1 = f1_score(y_test_c, class_preds)

print("\nCLASSIFICATION METRICS")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

# =========================================================
# PREMIUM OPTIMIZATION
# =========================================================

best_model_name = results_df.sort_values(
    by="R2",
    ascending=False
).iloc[0]["Model"]

best_model = trained_models[best_model_name]

# Claim probability
claim_prob = classifier.predict_proba(X_test_c)[:,1]

# Predicted severity
severity_predictions = best_model.predict(X_test)

# Example pricing formula
expense_loading = 500
profit_margin = 300

optimized_premium = (
    claim_prob[:len(severity_predictions)]
    * severity_predictions
    + expense_loading
    + profit_margin
)

premium_df = pd.DataFrame({
    "PredictedClaimProbability": claim_prob[:10],
    "PredictedSeverity": severity_predictions[:10],
    "OptimizedPremium": optimized_premium[:10]
})

print("\nOPTIMIZED PREMIUM EXAMPLES")
print(premium_df)

# =========================================================
# SHAP EXPLAINABILITY
# =========================================================

print("\nRUNNING SHAP ANALYSIS...")

# Use best model if tree-based
if best_model_name in ["Random Forest", "XGBoost"]:
    
    transformed_X_train = preprocessor.fit_transform(X_train)
    
    model_obj = best_model.named_steps["model"]
    
    explainer = shap.Explainer(model_obj)
    
    shap_values = explainer(transformed_X_train)
    
    shap.summary_plot(
        shap_values,
        transformed_X_train,
        show=False
    )
    
    plt.title("SHAP Feature Importance")
    plt.show()

# =========================================================
# FEATURE IMPORTANCE
# =========================================================

if best_model_name == "Random Forest":
    
    importances = model_obj.feature_importances_
    
elif best_model_name == "XGBoost":
    
    importances = model_obj.feature_importances_

else:
    
    importances = None

if importances is not None:
    
    feature_names = (
        numerical_cols
        +
        list(
            preprocessor.named_transformers_["cat"]
            .named_steps["encoder"]
            .get_feature_names_out(categorical_cols)
        )
    )
    
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values(
        by="Importance",
        ascending=False
    )
    
    print("\nTOP 10 IMPORTANT FEATURES")
    print(importance_df.head(10))
    
    plt.figure(figsize=(10,6))
    
    sns.barplot(
        data=importance_df.head(10),
        x="Importance",
        y="Feature"
    )
    
    plt.title("Top 10 Important Features")
    plt.show()

# =========================================================
# SAVE RESULTS
# =========================================================

results_df.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

print("\nPipeline Completed Successfully!")